# Elevate Labs — Task 1: Data Cleaning and Preprocessing
**Selected dataset:** Netflix Movies and TV Shows (Kaggle)

This notebook performs the required cleaning steps on one dataset and exports a cleaned CSV plus a summary report.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
RAW_FILE = ROOT / 'netflix_titles.csv'
CLEAN_FILE = ROOT / 'netflix_cleaned.csv'
REPORT_FILE = ROOT / 'cleaning_summary.txt'
DATA_URL = 'https://raw.githubusercontent.com/japnitahuja/netflix-data-analysis/main/netflix_titles.csv'


## 1. Load the raw dataset
Download `netflix_titles.csv` from Kaggle into this folder, or let this cell try the public mirror.

In [ ]:
if RAW_FILE.exists():
    raw = pd.read_csv(RAW_FILE)
else:
    raw = pd.read_csv(DATA_URL)
    raw.to_csv(RAW_FILE, index=False)

print('Raw dataset shape:', raw.shape)
display(raw.head())


## 2. Inspect data quality

In [ ]:
raw.info()
display(raw.isna().sum().to_frame('missing_values'))
print('Exact duplicate rows:', raw.duplicated().sum())


## 3. Clean and preprocess

In [ ]:
df = raw.copy()

# Clean headers.
df.columns = df.columns.astype(str).str.strip().str.lower().str.replace(r'\\s+', '_', regex=True)

# Trim strings and convert blanks to missing values.
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().replace('', pd.NA)

# Remove exact duplicate records.
duplicate_rows = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

# Fix data types.
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce', format='mixed')
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce').astype('Int64')

# Standardize categorical text.
df['type'] = df['type'].str.strip().str.title()
df['rating'] = df['rating'].str.strip().str.upper()

# Parse duration into number and unit.
parsed = df['duration'].str.extract(r'^\\s*(\\d+)\\s*(min|season|seasons)\\s*$', expand=True)
df['duration_value'] = pd.to_numeric(parsed[0], errors='coerce').astype('Int64')
df['duration_unit'] = parsed[1].str.lower().replace({'seasons': 'season'}).astype('string')
df['duration'] = df['duration'].str.replace(r'\\s+', ' ', regex=True)

# Fill selected descriptive columns. Do not invent dates/durations.
for col in ['director', 'cast', 'country', 'rating']:
    df[col] = df[col].fillna('Unknown')

print('Duplicates removed:', duplicate_rows)
print('Cleaned shape:', df.shape)
display(df.head())


## 4. Validate and export

In [ ]:
print('Duplicates remaining:', df.duplicated().sum())
display(df.isna().sum().to_frame('missing_after_cleaning'))
df.to_csv(CLEAN_FILE, index=False, date_format='%Y-%m-%d')

summary = [
    'ELEVATE LABS — TASK 1: DATA CLEANING SUMMARY',
    f'Raw dimensions: {raw.shape[0]} rows x {raw.shape[1]} columns',
    f'Exact duplicate rows removed: {duplicate_rows}',
    f'Cleaned dimensions: {df.shape[0]} rows x {df.shape[1]} columns',
    f'Duplicate rows remaining: {int(df.duplicated().sum())}',
    '', 'Missing values BEFORE cleaning:', raw.isna().sum().to_string(),
    '', 'Missing values AFTER cleaning:', df.isna().sum().to_string(),
    '', 'Changes: normalized headers and text; removed exact duplicates; converted date_added and release_year; standardized type/rating; parsed duration; filled selected descriptive fields with Unknown; preserved unknown dates/durations.'
]
REPORT_FILE.write_text('\\n'.join(summary), encoding='utf-8')
print('Saved:', CLEAN_FILE)
print('Saved:', REPORT_FILE)


## 5. Conclusion
The selected Netflix dataset has been cleaned and structured for future analysis. The summary report records the actual counts from the input file. This submission addresses Task 1 only and uses one dataset.